<a href="https://colab.research.google.com/github/Shifali1094/stance-detection/blob/main/notebooks/bertweet_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NLP Assignment 3 : Stance Detection with BERTweet**

## **1. Setup & Imports**
*Mount Drive, install dependencies, import libraries.*


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install libraries
!pip install -q transformers datasets evaluate accelerate emoji vncorenlp
!git clone https://github.com/Shifali1094/stance-detection.git

In [ ]:
# Imports
import pandas as pd
import numpy as np
import torch
from torch import nn
from transformers import Trainer
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from datasets import Dataset
from transformers import AutoTokenizer, TrainingArguments, AutoModelForSequenceClassification, DataCollatorWithPadding
import matplotlib.pyplot as plt
from scipy.special import softmax as scipy_softmax


from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

## **2. Data Preparation**

### **2.1. Load Dataset**

In [ ]:
# Detect environment
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Set project root
if IN_COLAB:
    PROJECT_ROOT = Path("/content/stance-detection")
else:
    PROJECT_ROOT = Path.cwd().parent

print(f"Running in Colab: {IN_COLAB}")
print(f"Project root: {PROJECT_ROOT}")

TEXT_COL = "text"
LABEL_COL = "label"

train_df = pd.read_csv(PROJECT_ROOT / "train.csv")
val_df   = pd.read_csv(PROJECT_ROOT / "val.csv")
test_df  = pd.read_csv(PROJECT_ROOT / "test.csv")

train_df = train_df[[TEXT_COL, LABEL_COL]].dropna()
val_df   = val_df[[TEXT_COL, LABEL_COL]].dropna()
test_df  = test_df[[TEXT_COL, LABEL_COL]].dropna()

train_df[TEXT_COL] = train_df[TEXT_COL].astype(str)
val_df[TEXT_COL]   = val_df[TEXT_COL].astype(str)
test_df[TEXT_COL]  = test_df[TEXT_COL].astype(str)

print(train_df.head())

### **2.2. Encode Labels**

In [ ]:
# Label encoding

labels = sorted(train_df[LABEL_COL].unique().tolist())

# Convert labels to normal Python values / strings
labels = [str(label) for label in labels]

train_df[LABEL_COL] = train_df[LABEL_COL].astype(str)
val_df[LABEL_COL] = val_df[LABEL_COL].astype(str)
test_df[LABEL_COL] = test_df[LABEL_COL].astype(str)

label2id = {label: int(i) for i, label in enumerate(labels)}
id2label = {int(i): str(label) for label, i in label2id.items()}

train_df["labels"] = train_df[LABEL_COL].map(label2id).astype(int)
val_df["labels"] = val_df[LABEL_COL].map(label2id).astype(int)
test_df["labels"] = test_df[LABEL_COL].map(label2id).astype(int)

print("Labels:", label2id)
print(train_df["labels"].value_counts())

### **2.3. Convert to Hugging Face Dataset**

In [ ]:
train_dataset = Dataset.from_pandas(
    train_df[[TEXT_COL, "labels"]]
)

val_dataset = Dataset.from_pandas(
    val_df[[TEXT_COL, "labels"]]
)

test_dataset = Dataset.from_pandas(
    test_df[[TEXT_COL, "labels"]]
)

### **2.4. Tokenise**

In [ ]:
MODEL_NAME = "vinai/bertweet-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False,
    normalization=True
)

def tokenize_function(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        padding=True,
        max_length=128
    )

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)

## **3. Model Setup**

### **3.1. Load BERTweet**

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

### **3.2. Define Metrics**

In [ ]:
def compute_metrics(eval_pred):

    logits, y_true = eval_pred

    y_pred = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    acc = accuracy_score(y_true, y_pred)

    return {
        "accuracy": acc,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    }

## **4. Training**

### **4.1. Training Arguuments**

In [ ]:
training_args = TrainingArguments(
    output_dir="./bertweet_results",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=25,
    report_to="none",
    seed=42
)

### **4.2. Class Weights & Weighted Trainer**

In [ ]:
# Compute inverse-frequency weights from training labels
class_counts = train_df["labels"].value_counts().sort_index().values
print("Class counts:", class_counts)

# Inverse frequency, then normalise so they sum to num_classes
weights = len(class_counts) / (class_counts * class_counts.sum())
class_weights_tensor = torch.tensor(weights, dtype=torch.float)
print("Class weights:", class_weights_tensor)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights_tensor.to(logits.device)
        )
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Weighted trainer created, class weights applied")

### **4.3. Run Training**

In [ ]:
train_result = trainer.train()

print(train_result)

## **5. Evaluation**

### **5.1. Test Set Results**

In [ ]:
test_results = trainer.evaluate(test_dataset)

print(test_results)

### **5.2. Classification Report**

In [ ]:
predictions = trainer.predict(test_dataset)

y_true = predictions.label_ids

y_pred = np.argmax(
    predictions.predictions,
    axis=1
)

target_names = [
    str(id2label[i])
    for i in range(len(labels))
]

print(
    classification_report(
        y_true,
        y_pred,
        target_names=target_names,
        zero_division=0
    )
)

## **6. Visualisation**

### **6.1. Per-class Precision / Recall / F1**

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred
)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=target_names
)
fig, ax = plt.subplots(
    figsize=(8, 6)
)
disp.plot(
    ax=ax,
    xticks_rotation=45
)
plt.title(
    "BERTweet Confusion Matrix"
)
plt.tight_layout()
plt.savefig(
    "bertweet_confusion_matrix.png",
    dpi=300
)
plt.show()

In [ ]:
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 150,
})
CLASS_NAMES = ["FAVOR", "NONE", "AGAINST"]
COLORS = ["#378ADD", "#888780", "#D85A30"]


In [ ]:
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, data, fmt, title in zip(
    axes,
    [cm, cm_norm],
    ["d", ".2f"],
    ["Raw counts", "Row-normalised (recall per class)"]
):
    sns.heatmap(
        data, annot=True, fmt=fmt, cmap="Blues",
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        linewidths=0.5, linecolor="white",
        cbar_kws={"shrink": 0.8}, ax=ax
    )
    ax.set_title(title, fontsize=12, fontweight="bold", pad=10)
    ax.set_xlabel("Predicted label", fontsize=11)
    ax.set_ylabel("True label", fontsize=11)
    ax.tick_params(axis="x", rotation=0)
    ax.tick_params(axis="y", rotation=0)

fig.suptitle("BERTweet  Confusion Matrix", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


### **6.1. Per-class Precision / Recall/ F1**

In [ ]:
report = classification_report(
    y_true, y_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

metrics   = ["precision", "recall", "f1-score"]
x         = np.arange(len(CLASS_NAMES))
bar_w     = 0.25
bar_color = ["#185FA5", "#1D9E75", "#EF9F27"]   # blue, teal, amber

fig, ax = plt.subplots(figsize=(8, 4.5))
for i, (metric, color) in enumerate(zip(metrics, bar_color)):
    vals = [report[cls][metric] for cls in CLASS_NAMES]
    bars = ax.bar(x + i * bar_w, vals, bar_w, label=metric.capitalize(),
                  color=color, alpha=0.9, zorder=3)
    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{v:.2f}", ha="center", va="bottom", fontsize=9
        )

ax.set_xticks(x + bar_w)
ax.set_xticklabels(CLASS_NAMES, fontsize=11)
ax.set_ylabel("Score", fontsize=11)
ax.set_ylim(0, 1.12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.1f}"))
ax.legend(frameon=False, fontsize=10)
ax.set_title("BERTweet — Per-class Precision, Recall & F1", fontsize=12, fontweight="bold")
ax.axhline(report["macro avg"]["f1-score"], color="gray", linewidth=1,
           linestyle="--", label="Macro F1")
ax.text(len(CLASS_NAMES) - 0.05,
        report["macro avg"]["f1-score"] + 0.01,
        f'Macro F1 = {report["macro avg"]["f1-score"]:.2f}',
        ha="right", va="bottom", color="gray", fontsize=9)
ax.grid(axis="y", linewidth=0.5, alpha=0.4, zorder=0)
plt.tight_layout()
plt.show()

### **6.2. Training & Validation Loss Curve**

In [ ]:
log_history = trainer.state.log_history

train_steps  = [e["step"] for e in log_history if "loss" in e]
train_losses = [e["loss"] for e in log_history if "loss" in e]
eval_steps   = [e["step"] for e in log_history if "eval_loss" in e]
eval_losses  = [e["eval_loss"] for e in log_history if "eval_loss" in e]

if train_steps:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(train_steps, train_losses, label="Train loss",
            color="#185FA5", linewidth=2, marker="o", markersize=3)
    if eval_steps:
        ax.plot(eval_steps, eval_losses, label="Val loss",
                color="#D85A30", linewidth=2, marker="s", markersize=5,
                linestyle="--")
    ax.set_xlabel("Step", fontsize=11)
    ax.set_ylabel("Loss", fontsize=11)
    ax.set_title("BERTweet Training & Validation Loss", fontsize=12, fontweight="bold")
    ax.legend(frameon=False, fontsize=10)
    ax.grid(linewidth=0.4, alpha=0.4)
    plt.tight_layout()
    plt.show()
else:
    print("No training loss logs found skipping loss curve.")

### **6.3. ROC Curves**

In [ ]:
predictions = trainer.predict(test_dataset)
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from scipy.special import softmax as scipy_softmax

try:
    probs = scipy_softmax(predictions.predictions, axis=1)
    y_true_bin = label_binarize(y_true, classes=list(range(len(labels))))

    fig, ax = plt.subplots(figsize=(7, 5))
    for i, (cls, color) in enumerate(zip(CLASS_NAMES, COLORS)):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], probs[:, i])
        roc_auc     = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f"{cls}  (AUC = {roc_auc:.2f})",
                color=color, linewidth=2)

    ax.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5)
    ax.set_xlabel("False positive rate", fontsize=11)
    ax.set_ylabel("True positive rate", fontsize=11)
    ax.set_title("BERTweet One-vs-Rest ROC Curves", fontsize=12, fontweight="bold")
    ax.legend(frameon=False, fontsize=10)
    ax.grid(linewidth=0.4, alpha=0.4)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"ROC curves skipped: {e}")

## **7. Save Outputs**

### **7.1. Predictions CSV**

In [ ]:
# Save predictions

test_output = test_df.copy()

test_output["predicted_label_id"] = y_pred

test_output["predicted_label"] = [
    id2label[int(i)]
    for i in y_pred
]

test_output["true_label"] = [
    id2label[int(i)]
    for i in y_true
]

test_output.to_csv(
    "bertweet_test_predictions.csv",
    index=False
)

print(
    "Predictions saved successfully"
)

test_output.head()

### **7.2. Model & Tokenizer**

In [ ]:
# Save final model and tokenizer

trainer.save_model(
    "./final_bertweet_model"
)

tokenizer.save_pretrained(
    "./final_bertweet_model"
)

print(
    "Model and tokenizer saved successfully"
)